In [ ]:
from openai import OpenAI
import json
import re
import os

# --- CONFIGURATION ---
api_keys_path = "../config/api_keys.json"
with open(api_keys_path, "r", encoding="utf-8") as f:
    api_key = json.load(f)["api_openai"]

client = OpenAI(api_key=api_key)

requirement_id = "11"

# --- FILE PATHS TO COMPARE ---
paths = [
    "/home/marisolbarrientosmoreno/Desktop/ER_2025/repo/Requirements_Change_for_Business_Process_Compliance/data/output_1st_execution/01_formalized_requirements/SIM_card_scenario/formalized_requirements_r" + requirement_id + ".json",
    "/home/marisolbarrientosmoreno/Desktop/ER_2025/repo/Requirements_Change_for_Business_Process_Compliance/data/output_2nd_execution/01_formalized_requirements/SIM_card_scenario/formalized_requirements_r" + requirement_id + ".json",
    "/home/marisolbarrientosmoreno/Desktop/ER_2025/repo/Requirements_Change_for_Business_Process_Compliance/data/output_3rd_execution/01_formalized_requirements/SIM_card_scenario/formalized_requirements_r" + requirement_id + ".json",
    "/home/marisolbarrientosmoreno/Desktop/ER_2025/repo/Requirements_Change_for_Business_Process_Compliance/data/output_4th_execution/01_formalized_requirements/SIM_card_scenario/formalized_requirements_r" + requirement_id + ".json",
    "/home/marisolbarrientosmoreno/Desktop/ER_2025/repo/Requirements_Change_for_Business_Process_Compliance/data/output_5th_execution/01_formalized_requirements/SIM_card_scenario/formalized_requirements_r" + requirement_id + ".json"
]

# --- SYSTEM PROMPT ---
system_prompt = """
You will receive multiple JSON arrays, each labeled with a unique name (e.g., iteration1, iteration2, iteration3, etc.).
Each array contains multiple requirements, each with fields such as id, precondition, norms, and temporal_validity.

- Do not compare entries *within* a single JSON array (e.g., do not compare r14_1 vs r14_2 inside iteration1).
- Only compare entire JSON arrays *against each other* (e.g., iteration1 vs iteration2).

Please do the following:

1. Compare each JSON array against every other array (pairwise).
2. Identify **any differences** between the arrays, including:
   - Changes in any field values across arrays
   - Added or missing entries (e.g., one array has 3 items, another has 2)
   - Any content mismatch for the same `id`
3. Present the results in a table with these columns:
   - Compared JSONs (e.g., iteration1 vs iteration2)
   - Are they identical? (Yes / No)
   - Field or path where the difference occurs (e.g., `norms[0].action.activities[0]`)
   - Description of the difference
4. If some JSON arrays are fully identical, group them together.

Paste your JSON arrays one after another and label them like:

iteration1:
[ ... ]

iteration2:
[ ... ]

Here are the names of the JSONs and JSON arrays to compare:
"""

# --- FORMAT INPUT FOR GPT ---
def extract_label(path):
    exec_match = re.search(r"output_(\d\w+)_execution", path)
    req_match = re.search(r"requirements_(r\d+)\.json", path)
    exec_label = exec_match.group(1) if exec_match else "unknown_exec"
    req_label = req_match.group(1) if req_match else "unknown_req"
    return f"{exec_label}_{req_label}"

input_sections = []

for path in paths:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        label = extract_label(path)
        input_sections.append(f"{label}:\n{json.dumps(data, indent=2, ensure_ascii=False)}")
    else:
        print(f"File not found: {path}")

# Combine everything
user_input = "\n\n".join(input_sections)

# --- QUERY GPT ---
def query_llm(system_prompt, user_input, model="gpt-4.1", temperature=0.0):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_input.strip()}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

# Run it
reply = query_llm(system_prompt, user_input)
print(reply)

# Save reply to file
output_file_path = "stability_results_r" + requirement_id + ".txt"
with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(reply)

print(f" LLM comparison results saved to: {output_file_path}")


In [ ]:
You are given:

1. A **ground truth deviation statement** that reflects the correct interpretation of a deviation resulting from a change in a requirement.
2. A list of **deviation JSONs**, each describing a deviation from the same requirement change.

Your task is to evaluate how well the meaning of each deviation (specifically the first deviation object, e.g., `*_d1`) aligns with the meaning of the ground truth.

### Ground Truth Deviation:
"{insert your ground truth deviation here}"

### For Each JSON, Assess the Following Semantically:

1. **Deviation Type Match**
   - Does the `deviation_type` express the same meaning as in the ground truth?
     (e.g., `over-compliance`, `non-compliance`, `under-compliance`, etc.)

2. **Semantic Equivalence of the Root Cause**
   - Does the JSON describe **the same reason** for the deviation as the ground truth?
   - Is the **core logic or obligation** captured correctly (e.g., an obligation was removed, but the process still enforces it)?

3. **Interpretation of the Updated Requirement**
   - Does the deviation show a **correct understanding** of what the new version of the requirement actually demands or no longer demands?

4. **Interpretation of the Process Behavior**
   - Does the deviation correctly identify what the process model still does, and why this is problematic (e.g., still enforces something no longer required)?

5. **Alignment of Process Model Reference**
   - Are the **process model elements** referenced in a way that supports the deviation claim (even if named differently)?
     Consider different ways of referring to the same concept (e.g., "Retract consent" vs "termination upon consent retraction").

6. **Semantic Alignment Score** (1–5)
   - 5 = Captures all key meanings of the ground truth
   - 3 = Partially aligned; core idea is there but not fully captured or expressed
   - 1 = Misunderstands or omits key aspects of the ground truth meaning

### Output Format per JSON:

- **JSON #N**
  - Deviation Type Match: Yes / No
  - Root Cause Match: Yes / Partial / No
  - Updated Requirement Interpretation: Correct / Partially Correct / Incorrect
  - Process Behavior Interpretation: Correct / Partially Correct / Incorrect
  - Process Model Reference: Aligned / Equivalent / Not Aligned
  - Semantic Alignment Score: X/5
  - **Summary Comment**: [Explain where the deviation meaning aligns or diverges from the ground truth.]

Now, evaluate each JSON deviation based on meaning, not writing style.
